# Baseline zero-shot (100 exemplos) com Ollama + log-probabilidades


In [ ]:
!nvidia-smi

In [ ]:
REPO_URL = "https://github.com/Luis-FO/Prompt-Engineering-for-KE.git"
BRANCH = "rui/colab"
MODEL = "qwen3.5:4b"

import os, json
os.chdir("/content")
if not os.path.isdir("kl"):
    !git clone -b {BRANCH} {REPO_URL} kl
else:
    !git -C /content/kl pull
os.chdir("/content/kl")
!ls 2_inference_module

In [ ]:
# Instalar o Ollama na máquina do Colab, arrancar o servidor e descarregar o modelo
!apt-get install -y -q zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, urllib.request
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("O servidor do Ollama não arrancou. Vê /content/ollama.log")
!ollama pull {MODEL}

In [ ]:
# -U garante a versão mais recente do langchain-ollama (as log-probabilidades precisam de uma versão recente)
!pip install -q -U langchain-ollama tqdm

In [ ]:
# Google Drive: guarda os resultados e permite retomar se o Colab desligar
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/kl_out_zero_shot_100_" + MODEL.replace(":", "_")

## 1. Teste com 3 prompts: o Ollama devolve log-probabilidades?

In [ ]:
!python 2_inference_module/run_inference_ollama_logprobs.py --model_name {MODEL} --limit 3 --out_dir /content/out_teste

In [ ]:
d = json.load(open("/content/out_teste/zero_shot_prompts_test.json"))
x = d[0]
print(x["generated_text"][:300])
print()
assert x["logprobs"], "O Ollama não devolveu log-probabilidades"
for e in x["logprobs"][:25]:
    alts = ", ".join(f"{t['token']!r}:{t['logprob']:.2f}" for t in e.get("top_logprobs", [])[:3])
    print(f"{e['token']!r:>14}  {e['logprob']:.3f}   | {alts}")
print("\nok: acesso às log-probabilidades confirmado,", len(x["logprobs"]), "tokens na primeira resposta")

## 2. Baseline: zero-shot, primeiros 100 exemplos

In [ ]:
!python 2_inference_module/run_inference_ollama_logprobs.py --model_name {MODEL} --limit 100 --out_dir {OUT}

## 3. Post-processamento e avaliação

In [ ]:
!python 2_inference_module/post_process_generated_string.py --input_folder {OUT} --out_dir {OUT}_pp
!python 3_evaluation_module/evaluation_reference_to_annotations/evaluate.py --input_dir {OUT}_pp --gold_file 1_data_module/1_data_preprocessing/RED-fm/test.jsonl --out_dir {OUT}_eval

In [ ]:
import statistics
d = json.load(open(f"{OUT}/zero_shot_prompts_test.json"))
medias = [sum(e["logprob"] for e in x["logprobs"]) / len(x["logprobs"]) for x in d if x["logprobs"]]
print("respostas:", len(d))
print("logprob média por resposta: média %.3f | mínimo %.3f | máximo %.3f" % (statistics.mean(medias), min(medias), max(medias)))